# XAUUSD classifier_v3 full training
Runs repository code only. The untouched test remains locked and is never loaded.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = 'https://github.com/Qauntify/qauntify_webV1.git'
REPO_DIR = '/content/qauntify_webV1'
REPO_BRANCH = 'main'
DATASET_ROOT = '/content/drive/MyDrive/Quantify/training_v3'
EXPERIMENT_DIR = '/content/drive/MyDrive/Quantify/experiments/classifiers_v3_full_001'
EXPECTED_DATASET_CHECKSUM = '7D281649106BDE4E457318EF807B1872FD06782FD9F972727AD773DFE024C5DF'
pathlib.Path(EXPERIMENT_DIR).parent.mkdir(parents=True, exist_ok=True)


In [ ]:
if not pathlib.Path(REPO_DIR, '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', pathlib.Path(REPO_DIR, 'requirements-training.txt')], check=True)


In [ ]:
import json
manifest_path = pathlib.Path(DATASET_ROOT, 'training_manifest.json')
assert manifest_path.is_file(), f'Missing frozen dataset manifest: {manifest_path}'
manifest = json.loads(manifest_path.read_text())
assert manifest['version'] == 'training_v3'
assert manifest['approval_status'] == 'approved_frozen'
assert manifest['dataset_checksum'] == EXPECTED_DATASET_CHECKSUM
assert manifest['untouched_test_locked'] is True
assert len(manifest['model_feature_columns']) == 81
assert pathlib.Path(DATASET_ROOT, 'dataset').is_dir()
assert pathlib.Path(DATASET_ROOT, 'walk_forward_assignments').is_dir()
print('Frozen training_v3 contract verified:', manifest['dataset_checksum'])


## Full run
This may take hours. Re-run this cell after a disconnection; `--resume` skips all completed model jobs.


In [ ]:
command = [sys.executable, '-m', 'ml.training.v3_cli',
    '--config', 'ml/configs/classifiers_v3.yaml',
    '--dataset-root', DATASET_ROOT,
    '--experiment-dir', EXPERIMENT_DIR]
if pathlib.Path(EXPERIMENT_DIR, 'run_state.json').is_file():
    command.append('--resume')
subprocess.run(command, cwd=REPO_DIR, check=True)


In [ ]:
result = json.loads(pathlib.Path(EXPERIMENT_DIR, 'experiment_manifest.json').read_text())
assert result['status'] == 'complete'
assert result['completed_jobs'] == result['expected_jobs'] == 60
assert result['untouched_test_accessed'] is False
print(json.dumps(result, indent=2))
